# Ejemplo programable: agente reactivo basado en modelo para movilidad

Este notebook implementa un agente reactivo basado en modelo para el caso de NYC Yellow Taxi. Mediante un replay histórico, cada zona mantiene un **estado interno persistente** que resume observaciones anteriores, incertidumbre y la última acción.

$$m_t=U(m_{t-1},a_{t-1},p_t), \qquad a_t=g(m_t).$$

**Objetivos de aprendizaje**

- distinguir percepción actual, estado del mundo, estado interno y acción;
- implementar un modelo de observación y un modelo de transición;
- comprobar que una misma percepción puede producir acciones distintas por la historia;
- representar incertidumbre cuando falta una observación;
- diferenciar reacción basada en modelo de planificación y búsqueda.

## 1. Alcance y límites

TLC registra viajes realizados y reportados. No registra demanda total, solicitudes no atendidas, tiempos de espera ni vehículos libres. Por ello, el estado interno de este agente representa **persistencia de actividad Yellow Taxi observada**, no demanda real ni necesidad comprobada de refuerzo.

Los archivos son históricos. En el replay, al cierre de la hora $h$ se revelan los registros de $h$ y el agente emite una recomendación simulada para $h+1$. Esta convención didáctica no convierte TLC en una fuente en tiempo real.

NOAA se omite intencionalmente para aislar el papel de la memoria. El agente utiliza únicamente actividad TLC reportada y no incorpora variables meteorológicas en sus reglas.

## 2. Arquitectura reactiva basada en modelo

| Componente | Función en el agente |
|---|---|
| Percepción | Revela la actividad de la zona en la hora actual y su referencia histórica. |
| Estado interno | Resume intensidad, persistencia, antigüedad, confianza y acción anterior. |
| Modelo de transición | Actualiza el estado con la nueva percepción o lo degrada si falta información. |
| Reglas condición-acción | Seleccionan una respuesta a partir del estado interno actualizado. |
| Memoria persistente | Conserva un estado separado para cada zona entre horas consecutivas. |

El agente continúa siendo reactivo: después de actualizar $m_t$, aplica reglas condición-acción locales. No genera sucesores, no mantiene una frontera y no compara caminos hacia un objetivo.

In [ ]:
from dataclasses import asdict, dataclass, replace

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

## 3. Datos y períodos del experimento

El período inicial de siete días fija una referencia por zona y hora del día. El período siguiente se reproduce cronológicamente como replay. El modelo no recalcula sus umbrales con el futuro del período evaluado.

In [ ]:
ZONA_HORARIA = "America/New_York"
INICIO_REFERENCIA = pd.Timestamp("2024-01-01 00:00:00", tz=ZONA_HORARIA)
FIN_REFERENCIA = pd.Timestamp("2024-01-08 00:00:00", tz=ZONA_HORARIA)
FIN_REPLAY = pd.Timestamp("2024-01-15 00:00:00", tz=ZONA_HORARIA)
URL_VIAJES = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
URL_ZONAS = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

print("Referencia:", INICIO_REFERENCIA, "→", FIN_REFERENCIA)
print("Replay:     ", FIN_REFERENCIA, "→", FIN_REPLAY)

In [ ]:
filtros = [
    ("tpep_pickup_datetime", ">=", INICIO_REFERENCIA.tz_localize(None).to_pydatetime()),
    ("tpep_pickup_datetime", "<", FIN_REPLAY.tz_localize(None).to_pydatetime()),
]
viajes = pd.read_parquet(
    URL_VIAJES,
    columns=["tpep_pickup_datetime", "PULocationID"],
    filters=filtros,
    engine="pyarrow",
)
zonas = pd.read_csv(URL_ZONAS).rename(
    columns={"LocationID": "PULocationID", "Borough": "borough", "Zone": "zona"}
)
zonas["PULocationID"] = pd.to_numeric(zonas["PULocationID"], errors="raise").astype("int64")
if zonas["PULocationID"].duplicated().any():
    raise ValueError("El catálogo de zonas no tiene una clave única")

viajes = viajes.loc[
    viajes["tpep_pickup_datetime"].notna()
    & viajes["PULocationID"].isin(zonas["PULocationID"])
].copy()
viajes["pickup_hora"] = (
    viajes["tpep_pickup_datetime"]
    .dt.tz_localize(ZONA_HORARIA, ambiguous="raise", nonexistent="raise")
    .dt.floor("h")
)
conteos = viajes.groupby(["pickup_hora", "PULocationID"], observed=True).size().rename("pickups").reset_index()
horas = pd.date_range(INICIO_REFERENCIA, FIN_REPLAY, freq="h", inclusive="left")
grilla = pd.MultiIndex.from_product(
    [horas, zonas["PULocationID"]], names=["pickup_hora", "PULocationID"]
).to_frame(index=False)
actividad = (
    grilla.merge(conteos, on=["pickup_hora", "PULocationID"], how="left", validate="one_to_one")
    .merge(zonas[["PULocationID", "borough", "zona"]], on="PULocationID", how="left", validate="many_to_one")
)
actividad["pickups"] = actividad["pickups"].fillna(0).astype("int64")
actividad["hora_dia"] = actividad["pickup_hora"].dt.hour
assert not actividad.duplicated(["pickup_hora", "PULocationID"]).any()
print(f"Viajes válidos: {len(viajes):,} | Filas zona-hora: {len(actividad):,}")

In [ ]:
datos_referencia = actividad.loc[actividad["pickup_hora"] < FIN_REFERENCIA].copy()
referencia = (
    datos_referencia.groupby(["PULocationID", "hora_dia"], as_index=False, observed=True)
    .agg(
        n_referencias=("pickups", "size"),
        mediana_pickups=("pickups", "median"),
        p75_pickups=("pickups", lambda serie: serie.quantile(0.75)),
    )
)
replay = (
    actividad.loc[actividad["pickup_hora"].between(FIN_REFERENCIA, FIN_REPLAY, inclusive="left")]
    .merge(referencia, on=["PULocationID", "hora_dia"], how="left", validate="many_to_one")
    .sort_values(["PULocationID", "pickup_hora"])
    .reset_index(drop=True)
)
assert datos_referencia["pickup_hora"].max() < FIN_REFERENCIA
assert referencia["n_referencias"].eq(7).all()
display(referencia.head())

## 4. PEAS y observabilidad parcial

| Componente | Implementación didáctica |
|---|---|
| **Performance** | Coherencia de transiciones y acciones, abstención, no repetición inmediata de alertas y trazabilidad |
| **Environment** | Replay histórico por zona-hora, actividad TLC reportada, referencia fija, capacidad simulada y responsable humano |
| **Actuators** | Mensajes `OBSERVAR`, `MONITOREAR`, `RECOMENDAR_REFUERZO` y falla segura `ABSTENERSE` |
| **Sensors** | Mecanismo lógico que revela una hora, valida la fila y forma una percepción |

El estado real incluye demanda, espera, vehículos y tráfico que no observamos. `pickups` es una percepción parcial. El estado interno no recupera mágicamente las variables ausentes: solo resume la historia disponible bajo un modelo declarado.

## 5. Modelo conceptual

Este agente no utiliza un modelo de *machine learning* entrenado. Se basa en un **modelo interno dinámico, determinista y declarado mediante reglas** que resume, para cada zona, la actividad observada hasta la hora actual. El modelo no reconstruye la demanda real: mantiene solo la información necesaria para las decisiones definidas en este ejemplo.

La transición general puede escribirse como:

$$m_t = T(m_{t-1}, p_t)$$

donde $m_{t-1}$ es el estado interno anterior, $p_t$ es la percepción recibida en la hora actual y $m_t$ es el nuevo estado. La función $T$ está implementada mediante reglas explícitas, no aprendidas automáticamente.

El programa separa cuatro operaciones:

1. **Observar:** clasificar la actividad actual respecto de mediana y percentil 75.
2. **Predecir:** trasladar el estado anterior una hora hacia adelante; si falta evidencia, la intensidad decae y aumenta su antigüedad.
3. **Corregir:** combinar la predicción con la observación mediante suavizado y actualizar persistencia y confianza.
4. **Reaccionar:** aplicar reglas al estado actualizado, sin explorar acciones futuras.

La acción previa solo modifica el estado del flujo de trabajo: después de `RECOMENDAR_REFUERZO` se aplica un breve enfriamiento. Como el agente emite mensajes y no ejecuta traslados, el modelo **no resta pickups ni supone un efecto físico**.

In [ ]:
@dataclass(frozen=True)
class PoliticaModelo:
    minimo_referencias: int = 5
    peso_observacion: float = 0.65
    decaimiento_sin_observacion: float = 0.85
    max_horas_sin_observacion: int = 1
    persistencia_para_refuerzo: int = 2
    enfriamiento_refuerzo_horas: int = 2
    version: str = "2.0-modelo"


@dataclass(frozen=True)
class PercepcionZona:
    zona_id: int
    hora_cerrada: pd.Timestamp
    pickups: object
    mediana_pickups: object
    p75_pickups: object
    n_referencias: int
    zona_valida: bool = True


@dataclass(frozen=True)
class EstadoInternoZona:
    zona_id: int
    hora_cerrada: pd.Timestamp
    intensidad_estimada: float
    nivel_estimado: str
    racha_alta: int
    observaciones_validas_consecutivas: int
    horas_sin_observacion: int
    confianza: str
    ultima_accion: str
    horas_desde_refuerzo: object


POLITICA = PoliticaModelo()
display(pd.Series(asdict(POLITICA), name="valor").to_frame())

### Componentes declarados en el código

El modelo queda definido por tres piezas complementarias:

1. **`PoliticaModelo`:** reúne los supuestos configurables: peso de la observación, decaimiento cuando falta información, persistencia mínima, referencias necesarias y duración del enfriamiento. Estos valores fueron elegidos con fines didácticos y no calibrados para una operación real.
2. **`PercepcionZona`:** representa la entrada disponible en una zona-hora: pickups, mediana, percentil 75, cantidad de referencias y validez de la zona.
3. **`EstadoInternoZona`:** representa la memoria persistente: intensidad estimada, nivel, racha alta, observaciones válidas, antigüedad, confianza y acción previa.

`interpretar_percepcion` funciona como **modelo de observación**. Primero valida la entrada y luego clasifica la actividad como `BAJA`, `MEDIA` o `ALTA`. También normaliza los pickups con la referencia histórica:

$$x_t = \frac{\text{pickups}_t}{\max(P_{75}, 1)}$$

Esta intensidad $x_t$ es una medida relativa de actividad observada. No representa demanda total ni probabilidad de necesitar refuerzo.

### Variables del estado interno

Para cada zona, el estado interno puede resumirse como:

$$m_t = (\text{intensidad}, \text{nivel}, \text{racha}, \text{validez}, \text{antigüedad}, \text{confianza}, \text{acción previa}, \text{enfriamiento})$$

| Variable | Función dentro del modelo |
|---|---|
| `intensidad_estimada` | Actividad relativa suavizada que combina la observación actual con la estimación anterior. |
| `nivel_estimado` | Clasificación del estado: `BAJA`, `MEDIA`, `ALTA_RECIENTE`, `ALTA_PERSISTENTE`, `PROYECTADO` o `DESCONOCIDO`. |
| `racha_alta` | Número de observaciones `ALTA` consecutivas. |
| `observaciones_validas_consecutivas` | Cantidad de horas consecutivas con información válida; interviene en la confianza. |
| `horas_sin_observacion` | Antigüedad de la última información válida. |
| `confianza` | Calidad cualitativa del estado: `ALTA`, `MEDIA` o `BAJA`. |
| `ultima_accion` | Última acción emitida por el agente. |
| `horas_desde_refuerzo` | Horas transcurridas desde `RECOMENDAR_REFUERZO`; controla el enfriamiento. |

El registro también contiene `zona_id`, que identifica a qué zona pertenece el estado, y `hora_cerrada`, que indica la hora representada y permite verificar la continuidad temporal. Son variables de identificación y control del estado, no indicadores de actividad.

En cambio, `pickups`, `mediana_pickups`, `p75_pickups`, `n_referencias` y `zona_valida` pertenecen a `PercepcionZona`: son entradas de la hora actual, no memoria persistente. Después de procesarlas, la información relevante queda resumida en `EstadoInternoZona`.

In [ ]:
def interpretar_percepcion(percepcion, politica=POLITICA):
    numeros = [percepcion.pickups, percepcion.mediana_pickups, percepcion.p75_pickups]
    valida = (
        percepcion.zona_valida
        and percepcion.n_referencias >= politica.minimo_referencias
        and all(pd.notna(valor) for valor in numeros)
        and percepcion.pickups >= 0
        and 0 <= percepcion.mediana_pickups <= percepcion.p75_pickups
    )
    if not valida:
        return {"valida": False, "nivel_observado": "NO_DISPONIBLE", "intensidad": np.nan}
    if percepcion.pickups <= percepcion.mediana_pickups:
        nivel = "BAJA"
    elif percepcion.pickups <= percepcion.p75_pickups:
        nivel = "MEDIA"
    else:
        nivel = "ALTA"
    escala = max(float(percepcion.p75_pickups), 1.0)
    return {
        "valida": True,
        "nivel_observado": nivel,
        "intensidad": float(percepcion.pickups) / escala,
    }

## 6. Programa del agente

El diccionario `estados` constituye la memoria persistente. Cada llamada a `paso` exige avanzar exactamente una hora para impedir reordenamientos silenciosos. Una observación faltante puede proyectarse durante un solo paso con confianza baja; después el agente se abstiene.

### Dónde se definen las reglas de transición

Las reglas de transición se implementan en `AgenteReactivoBasadoEnModelo.actualizar_estado`. Este método calcula $m_t$ a partir del estado anterior y de la percepción actual. Sus reglas son:

1. **Continuidad temporal:** si ya existe un estado, la nueva percepción debe corresponder exactamente a la hora siguiente.
2. **Primera observación:** cuando no existe estado anterior, la intensidad observada inicializa directamente la intensidad estimada.
3. **Observación válida:** combina la intensidad actual y la anterior mediante suavizado exponencial.
4. **Persistencia:** incrementa `racha_alta` ante observaciones `ALTA` consecutivas; en caso contrario la reinicia.
5. **Confianza:** una observación válida produce confianza `MEDIA`; dos o más consecutivas producen confianza `ALTA`.
6. **Observación faltante:** conserva temporalmente el estado con decaimiento, aumenta su antigüedad y reduce la confianza.
7. **Enfriamiento:** conserva cuántas horas transcurrieron desde la última recomendación de refuerzo.

Con una observación válida y un estado anterior, el suavizado aplicado es:

$$m_t = w x_t + (1-w)m_{t-1}$$

Como `peso_observacion = 0.65`, en este notebook se usa:

$$m_t = 0.65x_t + 0.35m_{t-1}$$

La observación actual recibe mayor peso, pero la historia reciente sigue influyendo. Si falta la observación, no se aplica esta corrección; se proyecta un paso mediante:

$$m_t = 0.85m_{t-1}$$

### Transición y decisión no son lo mismo

- `actualizar_estado()` contiene las **reglas de transición** y produce el nuevo estado interno.
- `decidir()` contiene las **reglas condición-acción** que seleccionan `OBSERVAR`, `MONITOREAR`, `RECOMENDAR_REFUERZO` o `ABSTENERSE`.
- `paso()` coordina ambas operaciones, registra el motivo y guarda el estado para la hora siguiente.

Esta separación permite identificar qué parte del programa representa la evolución interna y qué parte convierte ese estado en una respuesta visible.

### Significado de `capacidad_simulada`

La capacidad no se calcula a partir de los viajes TLC ni se actualiza mediante el modelo de transición. `capacidad_simulada` es una **entrada externa de control** que permite observar cómo cambia la acción ante disponibilidad o agotamiento hipotéticos. `paso()` usa el valor predeterminado `1`, y el replay histórico también lo fija explícitamente en `1`.

`decidir()` valida que la entrada sea un entero no negativo y la interpreta de forma binaria:

- `capacidad_simulada == 0`: la capacidad hipotética está agotada; incluso ante actividad alta persistente, el agente devuelve `MONITOREAR`.
- `capacidad_simulada > 0`: existe disponibilidad hipotética y el agente puede devolver `RECOMENDAR_REFUERZO` si se cumplen las demás condiciones.

Aunque el parámetro acepta enteros positivos, el programa no consume unidades después de una recomendación, no comparte un inventario entre zonas y no representa vehículos, rutas ni asignaciones. Por ello, funciona como un interruptor didáctico de disponibilidad y no como una simulación de flota. Tampoco forma parte de `EstadoInternoZona`: condiciona la acción actual, pero no es memoria persistente del agente.

In [ ]:
class AgenteReactivoBasadoEnModelo:
    ACCIONES = {"OBSERVAR", "MONITOREAR", "RECOMENDAR_REFUERZO", "ABSTENERSE"}

    def __init__(self, politica=POLITICA):
        self.politica = politica
        self.estados = {}
        self.bitacora = []

    def reiniciar(self):
        self.estados.clear()
        self.bitacora.clear()

    def actualizar_estado(self, percepcion):
        observacion = interpretar_percepcion(percepcion, self.politica)
        anterior = self.estados.get(percepcion.zona_id)
        if anterior is not None and percepcion.hora_cerrada != anterior.hora_cerrada + pd.Timedelta(hours=1):
            raise ValueError("Cada zona debe procesarse una vez por hora y en orden cronológico")

        intensidad_previa = 0.0 if anterior is None else anterior.intensidad_estimada
        if observacion["valida"]:
            if anterior is None:
                intensidad = observacion["intensidad"]
            else:
                w = self.politica.peso_observacion
                intensidad = w * observacion["intensidad"] + (1 - w) * intensidad_previa
            racha = (0 if anterior is None else anterior.racha_alta) + 1 if observacion["nivel_observado"] == "ALTA" else 0
            validas = (0 if anterior is None else anterior.observaciones_validas_consecutivas) + 1
            sin_observar = 0
            confianza = "ALTA" if validas >= 2 else "MEDIA"
            if observacion["nivel_observado"] == "ALTA":
                nivel = "ALTA_PERSISTENTE" if racha >= self.politica.persistencia_para_refuerzo else "ALTA_RECIENTE"
            else:
                nivel = observacion["nivel_observado"]
        else:
            sin_observar = 1 if anterior is None else anterior.horas_sin_observacion + 1
            intensidad = intensidad_previa * self.politica.decaimiento_sin_observacion
            racha = 0 if anterior is None else anterior.racha_alta
            validas = 0
            confianza = "BAJA"
            nivel = "PROYECTADO" if anterior is not None and sin_observar <= self.politica.max_horas_sin_observacion else "DESCONOCIDO"

        if anterior is None:
            horas_desde_refuerzo = None
        elif anterior.ultima_accion == "RECOMENDAR_REFUERZO":
            horas_desde_refuerzo = 1
        elif anterior.horas_desde_refuerzo is None:
            horas_desde_refuerzo = None
        else:
            horas_desde_refuerzo = anterior.horas_desde_refuerzo + 1

        return EstadoInternoZona(
            zona_id=percepcion.zona_id,
            hora_cerrada=percepcion.hora_cerrada,
            intensidad_estimada=float(intensidad),
            nivel_estimado=nivel,
            racha_alta=int(racha),
            observaciones_validas_consecutivas=int(validas),
            horas_sin_observacion=int(sin_observar),
            confianza=confianza,
            ultima_accion="NINGUNA" if anterior is None else anterior.ultima_accion,
            horas_desde_refuerzo=horas_desde_refuerzo,
        ), observacion

    def decidir(self, estado, observacion, capacidad_simulada):
        if isinstance(capacidad_simulada, bool) or not isinstance(capacidad_simulada, (int, np.integer)):
            raise TypeError("La capacidad simulada debe ser un entero no negativo")
        if capacidad_simulada < 0:
            raise ValueError("La capacidad simulada no puede ser negativa")
        if estado.nivel_estimado == "DESCONOCIDO":
            return "ABSTENERSE", "No existe una observación reciente suficiente"
        if not observacion["valida"]:
            return "MONITOREAR", "Estado proyectado un paso con confianza baja"
        if estado.nivel_estimado == "BAJA":
            return "OBSERVAR", "Actividad no supera la mediana histórica fijada"
        if estado.nivel_estimado in {"MEDIA", "ALTA_RECIENTE"}:
            return "MONITOREAR", "Actividad elevada todavía no persistente"
        en_enfriamiento = (
            estado.horas_desde_refuerzo is not None
            and estado.horas_desde_refuerzo < self.politica.enfriamiento_refuerzo_horas
        )
        if en_enfriamiento:
            return "MONITOREAR", "Existe una recomendación reciente pendiente de seguimiento"
        if capacidad_simulada == 0:
            return "MONITOREAR", "Actividad alta persistente, pero la capacidad simulada está agotada"
        return "RECOMENDAR_REFUERZO", "Actividad alta persistente, confianza suficiente y capacidad simulada disponible"

    def paso(self, percepcion, capacidad_simulada=1):
        estado, observacion = self.actualizar_estado(percepcion)
        accion, motivo = self.decidir(estado, observacion, capacidad_simulada)
        if accion == "RECOMENDAR_REFUERZO":
            estado = replace(estado, ultima_accion=accion, horas_desde_refuerzo=0)
        else:
            estado = replace(estado, ultima_accion=accion)
        self.estados[percepcion.zona_id] = estado
        registro = {
            "zona_id": percepcion.zona_id,
            "hora_cerrada": percepcion.hora_cerrada,
            "hora_objetivo": percepcion.hora_cerrada + pd.Timedelta(hours=1),
            "pickups": percepcion.pickups,
            "nivel_observado": observacion["nivel_observado"],
            **asdict(estado),
            "accion": accion,
            "motivo": motivo,
            "version_politica": self.politica.version,
            "modo_replay": True,
            "decision_humana": "PENDIENTE",
        }
        self.bitacora.append(registro)
        return registro

## 7. Prueba clave: misma percepción, distinta historia

Construimos dos historias controladas. En ambas, la segunda percepción contiene exactamente 20 pickups, mediana 10 y percentil 75 igual a 15. Solo cambia la hora anterior. El agente basado en modelo distingue un pico reciente de una señal persistente porque la decisión depende del estado interno acumulado.

In [ ]:
def percepcion_controlada(hora, pickups):
    return PercepcionZona(
        zona_id=1, hora_cerrada=pd.Timestamp(hora, tz=ZONA_HORARIA), pickups=pickups,
        mediana_pickups=10, p75_pickups=15, n_referencias=7,
    )


actual = percepcion_controlada("2024-01-08 09:00", 20)
agente_pico = AgenteReactivoBasadoEnModelo()
agente_pico.paso(percepcion_controlada("2024-01-08 08:00", 8))
salida_pico = agente_pico.paso(actual)

agente_persistencia = AgenteReactivoBasadoEnModelo()
agente_persistencia.paso(percepcion_controlada("2024-01-08 08:00", 18))
salida_persistencia = agente_persistencia.paso(actual)

comparacion_historias = pd.DataFrame([
    {"historia": "hora anterior baja", **salida_pico},
    {"historia": "hora anterior alta", **salida_persistencia},
])[["historia", "pickups", "nivel_observado", "racha_alta", "nivel_estimado", "confianza", "accion"]]
display(comparacion_historias)
assert salida_pico["pickups"] == salida_persistencia["pickups"] == 20
assert salida_pico["accion"] == "MONITOREAR"
assert salida_persistencia["accion"] == "RECOMENDAR_REFUERZO"

## 8. La acción anterior también entra al modelo

Una recomendación no se interpreta como un cambio físico de la movilidad. Solo abre un período de seguimiento en la representación del agente. Durante la hora siguiente, una condición todavía alta se monitorea en vez de duplicar inmediatamente el mismo mensaje.

In [ ]:
salida_10 = agente_persistencia.paso(percepcion_controlada("2024-01-08 10:00", 22))
salida_11 = agente_persistencia.paso(percepcion_controlada("2024-01-08 11:00", 21))
display(pd.DataFrame([salida_persistencia, salida_10, salida_11])[[
    "hora_cerrada", "racha_alta", "horas_desde_refuerzo", "accion", "motivo"
]])
assert salida_10["horas_desde_refuerzo"] == 1
assert salida_10["accion"] == "MONITOREAR"
assert salida_11["horas_desde_refuerzo"] == 0
assert salida_11["accion"] == "RECOMENDAR_REFUERZO"

## 9. Observación faltante e incertidumbre

Cuando falta una lectura, el modelo conserva temporalmente una estimación con decaimiento. La procedencia queda marcada como `PROYECTADO` y la confianza baja. Esta tolerancia dura una hora; una segunda ausencia obliga a `ABSTENERSE`. Proyectar no equivale a observar ni autoriza un refuerzo.

In [ ]:
agente_faltantes = AgenteReactivoBasadoEnModelo()
agente_faltantes.paso(percepcion_controlada("2024-01-08 08:00", 18))
observado = agente_faltantes.paso(percepcion_controlada("2024-01-08 09:00", 20))
faltante_1 = replace(percepcion_controlada("2024-01-08 10:00", 0), pickups=None)
faltante_2 = replace(percepcion_controlada("2024-01-08 11:00", 0), pickups=None)
proyectado = agente_faltantes.paso(faltante_1)
desconocido = agente_faltantes.paso(faltante_2)
display(pd.DataFrame([observado, proyectado, desconocido])[[
    "hora_cerrada", "nivel_observado", "intensidad_estimada",
    "horas_sin_observacion", "confianza", "nivel_estimado", "accion"
]])
assert proyectado["intensidad_estimada"] < observado["intensidad_estimada"]
assert proyectado["nivel_estimado"] == "PROYECTADO" and proyectado["accion"] == "MONITOREAR"
assert desconocido["nivel_estimado"] == "DESCONOCIDO" and desconocido["accion"] == "ABSTENERSE"

## 10. Replay con observaciones TLC reales

Seleccionamos reproduciblemente la primera zona que presenta al menos dos horas altas consecutivas. La selección solo sirve para asegurar que la demostración contenga una transición hacia `ALTA_PERSISTENTE`; no afirma que sea la zona más necesitada.

In [ ]:
replay["alta_observada"] = replay["pickups"].gt(replay["p75_pickups"])
replay["alta_anterior"] = replay.groupby("PULocationID", observed=True)["alta_observada"].shift(fill_value=False)
candidatos = replay.loc[replay["alta_observada"] & replay["alta_anterior"]]
if candidatos.empty:
    raise RuntimeError("No se encontró una zona con dos horas altas consecutivas")
zona_demo_id = int(candidatos.sort_values(["pickup_hora", "PULocationID"]).iloc[0]["PULocationID"])
serie_demo = replay.loc[replay["PULocationID"].eq(zona_demo_id)].sort_values("pickup_hora").copy()
nombre_demo = serie_demo["zona"].iloc[0]
print(f"Zona demostrativa: {zona_demo_id} - {nombre_demo} | Horas: {len(serie_demo)}")

In [ ]:
def percepcion_desde_fila(fila):
    return PercepcionZona(
        zona_id=int(fila.PULocationID),
        hora_cerrada=fila.pickup_hora,
        pickups=int(fila.pickups),
        mediana_pickups=float(fila.mediana_pickups),
        p75_pickups=float(fila.p75_pickups),
        n_referencias=int(fila.n_referencias),
        zona_valida=pd.notna(fila.zona),
    )


agente_real = AgenteReactivoBasadoEnModelo()
registros_reales = []
for fila in serie_demo.itertuples(index=False):
    percepcion = percepcion_desde_fila(fila)
    registro = agente_real.paso(percepcion, capacidad_simulada=1)
    registros_reales.append(registro)
bitacora_real = pd.DataFrame(registros_reales)
display(bitacora_real["accion"].value_counts().rename_axis("accion").rename("horas").to_frame())
display(bitacora_real.loc[bitacora_real["accion"].ne("OBSERVAR"), [
    "hora_cerrada", "pickups", "nivel_observado", "racha_alta",
    "nivel_estimado", "accion", "motivo"
]].head(15))

## 11. Invariantes y causalidad temporal

El modelo solo puede depender del prefijo ya procesado. Alteramos pickups posteriores a un corte y comprobamos que las decisiones anteriores no cambian. También verificamos que una percepción fuera de orden sea rechazada.

In [ ]:
assert bitacora_real["accion"].isin(AgenteReactivoBasadoEnModelo.ACCIONES).all()
assert bitacora_real["hora_objetivo"].eq(bitacora_real["hora_cerrada"] + pd.Timedelta(hours=1)).all()
assert bitacora_real["modo_replay"].all()
assert bitacora_real["decision_humana"].eq("PENDIENTE").all()
assert bitacora_real.loc[bitacora_real["accion"].eq("RECOMENDAR_REFUERZO"), "nivel_estimado"].eq("ALTA_PERSISTENTE").all()
assert bitacora_real["racha_alta"].ge(0).all()
assert bitacora_real["horas_sin_observacion"].ge(0).all()
assert bitacora_real["accion"].eq("RECOMENDAR_REFUERZO").any()

corte = serie_demo["pickup_hora"].iloc[47]
futuro_alterado = serie_demo.copy()
futuro_alterado.loc[futuro_alterado["pickup_hora"].gt(corte), "pickups"] += 10_000

def ejecutar_hasta(tabla, hora_limite):
    agente = AgenteReactivoBasadoEnModelo()
    salidas = []
    for fila in tabla.loc[tabla["pickup_hora"].le(hora_limite)].sort_values("pickup_hora").itertuples(index=False):
        salidas.append(agente.paso(percepcion_desde_fila(fila)))
    return pd.DataFrame(salidas)

control = ejecutar_hasta(serie_demo, corte)
alterado = ejecutar_hasta(futuro_alterado, corte)
columnas_decision = ["hora_cerrada", "pickups", "intensidad_estimada", "racha_alta", "nivel_estimado", "accion"]
pd.testing.assert_frame_equal(control[columnas_decision], alterado[columnas_decision])

agente_orden = AgenteReactivoBasadoEnModelo()
agente_orden.paso(percepcion_controlada("2024-01-08 09:00", 20))
try:
    agente_orden.paso(percepcion_controlada("2024-01-08 08:00", 18))
    raise AssertionError("El agente aceptó una percepción fuera de orden")
except ValueError as error:
    assert "orden cronológico" in str(error)

print("Invariantes: OK. El futuro alterado no cambió decisiones anteriores y el orden temporal se hizo cumplir.")

## 12. Conexión con representación y búsqueda

| Concepto | En este notebook |
|---|---|
| Estado del mundo | Movilidad real, demanda, espera, flota y tráfico, parcialmente desconocidos |
| Percepción | Pickups revelados para una zona-hora y su referencia histórica |
| Estado interno $m_t$ | Intensidad suavizada, persistencia, antigüedad, confianza y acción previa |
| Modelo de transición | Reglas que conservan, corrigen o degradan esas variables al avanzar una hora |
| Nodo de búsqueda | No existe en este programa; requeriría padre, acción, costo y profundidad |

El estado interno es suficiente solo para las reglas declaradas. Si quisiéramos elegir origen, ruta y secuencia de traslados, necesitaríamos representar vehículos, conexiones, tiempo y costos, formular $S,A,T,s_0,G,c$ y aplicar búsqueda. Añadir memoria no convierte automáticamente al agente en planificador.

## 13. Lectura crítica y extensiones

El modelo aporta memoria y una respuesta explícita ante observación parcial, pero también introduce supuestos: el peso de suavizado, el decaimiento, la persistencia mínima y el enfriamiento no fueron calibrados para una operación real. Un modelo incorrecto puede acumular error y dar una apariencia injustificada de conocimiento.

Las salidas mantienen `decision_humana = PENDIENTE`. `RECOMENDAR_REFUERZO` continúa siendo un mensaje para revisión, no un traslado ni una predicción de reducción de espera.

**Actividades de extensión**

1. Cambiar `persistencia_para_refuerzo` y explicar qué errores reduce y cuáles aumenta.
2. Diseñar una confianza numérica que dependa de antigüedad y cobertura.
3. Comparar conservar la racha durante un faltante frente a reiniciarla.
4. Proponer variables necesarias para pasar de reacción local a búsqueda de una ruta de refuerzo.
5. Identificar una prueba que detecte deriva del modelo sin afirmar causalidad.